# Claims audit

Static checks over the notebooks and the README. They do not re-run the analysis — that
is still *Run All* — but they catch the defects that survive a careful read:

- a code cell that references a name no earlier cell defines (this is how `sw` broke the
  basket section of `EDA.ipynb`)
- a code cell that does not compile
- a README link, image or notebook reference that does not resolve
- a `§N.M` section reference pointing at a heading that no longer exists
- a notebook that no longer parses as JSON

Run from inside `notebooks/`. Nothing here reads the workbook, so it is fast and works on
a fresh clone without `data/`.


In [ ]:
import ast
import builtins
import json
import re
from pathlib import Path

from IPython.core.inputtransformer2 import TransformerManager

HERE = Path.cwd()
REPO = HERE.parent
transform = TransformerManager().transform_cell
failures = []


def code_cells(nb):
    return [(i, "".join(c["source"]))
            for i, c in enumerate(nb["cells"]) if c["cell_type"] == "code"]


print(f"auditing {REPO.resolve()}")
print("notebooks:", ", ".join(p.name for p in sorted(HERE.glob("*.ipynb"))))


## 1. Notebooks parse and every code cell compiles

Syntax errors and corrupt JSON are the cheap failures; `%matplotlib inline` and `!pip`
style lines are transformed the way Jupyter would before compiling, so magics do not
produce false positives.


In [ ]:
parsed = {}
for path in sorted(HERE.glob("*.ipynb")):
    try:
        parsed[path] = json.loads(path.read_text())
    except json.JSONDecodeError as exc:
        failures.append(f"{path.name}: not valid JSON ({exc})")

for path, nb in parsed.items():
    bad = []
    for i, src in code_cells(nb):
        try:
            compile(transform(src), f"{path.name}[cell {i}]", "exec")
        except SyntaxError as exc:
            bad.append(f"cell {i}: {exc.msg} (line {exc.lineno})")
    failures.extend(f"{path.name}: {b}" for b in bad)
    status = "ok" if not bad else f"{len(bad)} cell(s) fail"
    print(f"  {path.name:30s} {len(nb['cells']):>3} cells   {status}")


## 2. No cell uses a name that nothing defines

Cells run top to bottom in a shared kernel, so a name is available if any earlier cell
bound it. The check collects what each cell binds — assignments, imports, `def`, `for`
targets, comprehension variables, `with ... as`, `except ... as` — and flags loads that
nothing before it provides.

It is deliberately conservative: it does not follow values into functions, so a global
read inside a `def` is checked when the function is defined, not when it is called. That
is the failure mode it is meant to catch, and it catches nothing else.


In [ ]:
def undefined_names(nb):
    builtin_names = set(dir(builtins)) | {"__builtins__", "get_ipython", "display"}
    defined, problems = set(builtin_names), []

    for i, src in code_cells(nb):
        tree = ast.parse(transform(src))
        local = {n.id for n in ast.walk(tree)
                 if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Store)}
        local |= {a.asname or a.name.split(".")[0]
                  for n in ast.walk(tree)
                  if isinstance(n, (ast.Import, ast.ImportFrom)) for a in n.names}
        local |= {n.name for n in ast.walk(tree)
                  if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))}
        local |= {a.arg for n in ast.walk(tree)
                  if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.Lambda))
                  for a in n.args.posonlyargs + n.args.args + n.args.kwonlyargs}
        local |= {h.optional_vars.id for n in ast.walk(tree)
                  if isinstance(n, (ast.With, ast.AsyncWith))
                  for h in n.items if isinstance(h.optional_vars, ast.Name)}
        local |= {n.name for n in ast.walk(tree)
                  if isinstance(n, ast.ExceptHandler) and n.name}
        local |= {name for n in ast.walk(tree)
                  if isinstance(n, (ast.Global, ast.Nonlocal)) for name in n.names}

        loaded = {n.id for n in ast.walk(tree)
                  if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Load)}
        problems.extend((i, name) for name in sorted(loaded - defined - local))
        defined |= local

    return problems


for path, nb in parsed.items():
    problems = undefined_names(nb)
    failures.extend(f"{path.name} cell {i}: `{name}` is used before anything defines it"
                    for i, name in problems)
    print(f"  {path.name:30s} {'ok' if not problems else str(len(problems)) + ' problem(s)'}")


## 3. Links and figures resolve

Relative links are resolved against the file that contains them: notebook links against
`notebooks/`, README links against the repository root.


In [ ]:
LINK = re.compile(r"!?\[[^\]]*\]\(([^)]+)\)")


def check_links(path, base):
    found = 0
    for target in LINK.findall(path.read_text()):
        if target.startswith(("http://", "https://", "#", "mailto:")):
            continue
        target = target.split("#")[0]
        if not target:
            continue
        found += 1
        if not (base / target).exists():
            failures.append(f"{path.name}: missing link target {target}")
    return found


readme = REPO / "README.md"
print(f"  README.md: {check_links(readme, REPO)} local link(s)")

for path in parsed:
    nb_links = 0
    for cell in parsed[path]["cells"]:
        if cell["cell_type"] != "markdown":
            continue
        for target in LINK.findall("".join(cell["source"])):
            if target.startswith(("http://", "https://", "#", "mailto:")):
                continue
            nb_links += 1
            if not (path.parent / target.split("#")[0]).exists():
                failures.append(f"{path.name}: missing link target {target}")
    print(f"  {path.name:30s} {nb_links} local link(s)")

figures = sorted((REPO / "figures").glob("*.png"))
print(f"  figures/: {len(figures)} PNG(s), "
      f"{min(f.stat().st_size for f in figures) // 1024}-"
      f"{max(f.stat().st_size for f in figures) // 1024} KB")
if not figures:
    failures.append("figures/: no PNGs committed")


## 4. Section references point at real headings

The prose cites sections as `§5.2`, and the README does the same. Renumbering a notebook
without updating the citations leaves a reader hunting for a heading that is not there.
This check pulls heading numbers out of `EDA.ipynb` and confirms every `§` reference
resolves.


In [ ]:
HEADING = re.compile(r"^#{2,4}\s+(\d+(?:\.\d+)?)[.\s]", re.MULTILINE)

eda = parsed[HERE / "EDA.ipynb"]
headings = {num for cell in eda["cells"] if cell["cell_type"] == "markdown"
            for num in HEADING.findall("".join(cell["source"]))}
print("  EDA headings:", ", ".join(sorted(headings, key=lambda s: [int(p) for p in s.split(".")])))

sources = {"README.md": readme.read_text()}
for path, nb in parsed.items():
    sources[path.name] = "\n".join("".join(c["source"]) for c in nb["cells"]
                                   if c["cell_type"] == "markdown")

refs = {}
for name, body in sources.items():
    for ref in re.findall(r"§\s*(\d+(?:\.\d+)?)", body):
        refs.setdefault(ref, set()).add(name)

for ref in sorted(refs):
    ok = ref in headings
    print(f"  §{ref:<5s} {','.join(sorted(refs[ref])):45s} {'ok' if ok else 'NO SUCH HEADING'}")
    if not ok:
        failures.append(f"§{ref} referenced by {', '.join(sorted(refs[ref]))} "
                        f"but EDA.ipynb has no such heading")


## 5. Result

Every check prints as it runs; the cell below is the gate. A non-empty list here means the
repository ships something a careful reader would trip over.


In [ ]:
print(f"{len(failures)} failure(s)")

for failure in sorted(set(failures)):
    print("  -", failure)

assert not failures, (
    f"{len(failures)} static check(s) failed — see the list above. "
    "Numbers are not checked here; re-run EDA.ipynb and compare the printed values "
    "with the prose."
)

print("\nall static checks passed")


## What this does not check

- **Numbers.** The audit does not recompute a single statistic. Re-running `EDA.ipynb`
  and comparing the printed values to the prose is still the only check that catches a
  stale claim, and it is the standard the notebooks are written to.
- **Whether a test is the right test.** The HAC lag, the multiple-comparison correction
  and the bootstrap unit are judgement calls. The audit confirms the names resolve, not
  that the inference is sound.
- **The workbook.** Nothing here needs `data/`, so a fresh clone can run it. The price is
  that a changed dataset — or a changed cell — is invisible to these checks.
